In [1]:
import numpy as np
import pandas as pd
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score, classification_report, precision_recall_curve
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [2]:
print("Loading data...")
df = pd.read_csv('application_train.csv')
print(f"  application_train : {df.shape}")

Loading data...
  application_train : (307511, 122)


In [5]:
try:
    # ── Load bureau_balance and aggregate to SK_ID_BUREAU level ──
    bureau = pd.read_csv('bureau.csv')
    print(f"  bureau            : {bureau.shape}")

    try:
        bb = pd.read_csv('bureau_balance.csv')
        print(f"  bureau_balance    : {bb.shape}")

        bb_agg = bb.groupby('SK_ID_BUREAU').agg(
            BB_MONTHS_COUNT     = ('MONTHS_BALANCE', 'count'),
            BB_STATUS_C_COUNT   = ('STATUS', lambda x: (x == 'C').sum()),
            BB_STATUS_X_COUNT   = ('STATUS', lambda x: (x == 'X').sum()),
            BB_STATUS_BAD_COUNT = ('STATUS', lambda x: x.isin(['1','2','3','4','5']).sum()),
        ).reset_index()

        # Compute bad rate per loan before merging
        bb_agg['BB_BAD_RATE'] = (
            bb_agg['BB_STATUS_BAD_COUNT'] / (bb_agg['BB_MONTHS_COUNT'] + 1)
        )

        # ── KEY: merge bureau_balance INTO bureau on SK_ID_BUREAU ──
        bureau = bureau.merge(bb_agg, on='SK_ID_BUREAU', how='left')
        print(f"  bureau after bb merge: {bureau.shape}")

    except FileNotFoundError:
        print("  bureau_balance.csv not found — skipping")

    # ── NOW aggregate to SK_ID_CURR level (BB columns included if available) ──
    bureau_agg = bureau.groupby('SK_ID_CURR').agg(
        BUREAU_LOAN_COUNT          = ('SK_ID_BUREAU',        'count'),
        BUREAU_ACTIVE_COUNT        = ('CREDIT_ACTIVE',       lambda x: (x == 'Active').sum()),
        BUREAU_CLOSED_COUNT        = ('CREDIT_ACTIVE',       lambda x: (x == 'Closed').sum()),
        BUREAU_AMT_CREDIT_SUM      = ('AMT_CREDIT_SUM',      'sum'),
        BUREAU_AMT_CREDIT_DEBT_SUM = ('AMT_CREDIT_SUM_DEBT', 'sum'),
        BUREAU_AMT_CREDIT_OVERDUE  = ('AMT_CREDIT_SUM_OVERDUE', 'sum'),
        BUREAU_DAYS_CREDIT_MEAN    = ('DAYS_CREDIT',         'mean'),
        BUREAU_DAYS_CREDIT_MAX     = ('DAYS_CREDIT',         'max'),
        BUREAU_DAYS_ENDDATE_MEAN   = ('DAYS_CREDIT_ENDDATE', 'mean'),
        BUREAU_CNT_PROLONG_SUM     = ('CNT_CREDIT_PROLONG',  'sum'),
        # bureau_balance roll-ups (NaN if bureau_balance.csv was missing)
        BB_MONTHS_COUNT_MEAN       = ('BB_MONTHS_COUNT',     'mean'),
        BB_BAD_COUNT_MEAN          = ('BB_STATUS_BAD_COUNT', 'mean'),
        BB_BAD_COUNT_SUM           = ('BB_STATUS_BAD_COUNT', 'sum'),
        BB_BAD_RATE_MEAN           = ('BB_BAD_RATE',         'mean'),
        BB_BAD_RATE_MAX            = ('BB_BAD_RATE',         'max'),
    ).reset_index()

    # Derived ratios
    bureau_agg['BUREAU_DEBT_CREDIT_RATIO'] = (
        bureau_agg['BUREAU_AMT_CREDIT_DEBT_SUM'] /
        (bureau_agg['BUREAU_AMT_CREDIT_SUM'] + 1)
    )
    bureau_agg['BUREAU_ACTIVE_RATIO'] = (
        bureau_agg['BUREAU_ACTIVE_COUNT'] /
        (bureau_agg['BUREAU_LOAN_COUNT'] + 1)
    )

    df = df.merge(bureau_agg, on='SK_ID_CURR', how='left')
    print(f"  After bureau merge: {df.shape}")

except FileNotFoundError:
    print("  bureau.csv not found — skipping")


  bureau            : (1716428, 17)
  bureau_balance    : (27299925, 3)
  bureau after bb merge: (1716428, 22)
  After bureau merge: (307511, 139)


In [6]:
try:
    pos = pd.read_csv('POS_CASH_balance.csv')
    pos_agg = pos.groupby('SK_ID_CURR').agg(
        POS_MONTHS_BALANCE_MEAN = ('MONTHS_BALANCE',        'mean'),
        POS_CNT_INSTALMENT_MEAN = ('CNT_INSTALMENT',        'mean'),
        POS_SK_DPD_MAX          = ('SK_DPD',                'max'),
        POS_SK_DPD_DEF_SUM      = ('SK_DPD_DEF',            'sum'),
        POS_COMPLETED_COUNT     = ('NAME_CONTRACT_STATUS',
                                   lambda x: (x=='Completed').sum()),
    ).reset_index()
    df = df.merge(pos_agg, on='SK_ID_CURR', how='left')
    print(f"After POS merge: {df.shape}")
except FileNotFoundError:
    print("POS_CASH_balance.csv not found — skipping")

After POS merge: (307511, 144)


In [7]:
try:
    prev = pd.read_csv('previous_application.csv')
    print(f"  previous_application: {prev.shape}")

    prev_agg = prev.groupby('SK_ID_CURR').agg(
        PREV_APP_COUNT             = ('SK_ID_PREV',           'count'),
        PREV_APPROVED_COUNT        = ('NAME_CONTRACT_STATUS', lambda x: (x == 'Approved').sum()),
        PREV_REFUSED_COUNT         = ('NAME_CONTRACT_STATUS', lambda x: (x == 'Refused').sum()),
        PREV_AMT_CREDIT_MEAN       = ('AMT_CREDIT',           'mean'),
        PREV_AMT_ANNUITY_MEAN      = ('AMT_ANNUITY',          'mean'),
        PREV_AMT_DOWN_PAYMENT_MEAN = ('AMT_DOWN_PAYMENT',     'mean'),
        PREV_DAYS_DECISION_MEAN    = ('DAYS_DECISION',        'mean'),
        PREV_DAYS_DECISION_MIN     = ('DAYS_DECISION',        'min'),
        PREV_RATE_DOWN_PAYMENT_MEAN= ('RATE_DOWN_PAYMENT',    'mean'),
        PREV_AMT_CREDIT_MAX   = ('AMT_CREDIT',    'max'),
        PREV_AMT_CREDIT_MIN   = ('AMT_CREDIT',    'min'),
        PREV_DAYS_LAST_DUE_MEAN = ('DAYS_LAST_DUE', 'mean'),
        PREV_RATE_INT_MEAN    = ('RATE_INTEREST_PRIMARY', 'mean'),
    ).reset_index()

    # Approval rate — key signal
    prev_agg['PREV_APPROVAL_RATE'] = (
        prev_agg['PREV_APPROVED_COUNT'] /
        (prev_agg['PREV_APP_COUNT'] + 1)
    )

    df = df.merge(prev_agg, on='SK_ID_CURR', how='left')
    print(f"  After prev merge  : {df.shape}")

except FileNotFoundError:
    print("  previous_application.csv not found — skipping")

  previous_application: (1670214, 37)
  After prev merge  : (307511, 158)


In [8]:
try:
    inst = pd.read_csv('installments_payments.csv')
    print(f"  installments_payments: {inst.shape}")

    # ── Compute DPD and DBD before aggregating ──
    # DPD (Days Past Due)  = paid AFTER due date → positive = late
    # DBD (Days Before Due) = paid BEFORE due date → positive = early
    inst['DPD'] = (inst['DAYS_ENTRY_PAYMENT'] - inst['DAYS_INSTALMENT']).clip(lower=0)
    inst['DBD'] = (inst['DAYS_INSTALMENT'] - inst['DAYS_ENTRY_PAYMENT']).clip(lower=0)

    # Payment difference: negative = underpaid, positive = overpaid
    inst['PAYMENT_DIFF']  = inst['AMT_INSTALMENT'] - inst['AMT_PAYMENT']

    # Payment ratio: how much of the instalment was actually paid
    inst['PAYMENT_RATIO'] = inst['AMT_PAYMENT'] / (inst['AMT_INSTALMENT'] + 1)

    inst_agg = inst.groupby('SK_ID_CURR').agg(
        INST_COUNT              = ('SK_ID_PREV',    'count'),
        INST_AMT_PAYMENT_SUM    = ('AMT_PAYMENT',   'sum'),
        INST_AMT_PAYMENT_MEAN   = ('AMT_PAYMENT',   'mean'),
        INST_PAYMENT_DIFF_MEAN  = ('PAYMENT_DIFF',  'mean'),  # avg underpayment
        INST_PAYMENT_DIFF_MAX   = ('PAYMENT_DIFF',  'max'),   # worst underpayment
        INST_PAYMENT_RATIO_MEAN = ('PAYMENT_RATIO', 'mean'),  # avg payment completeness
        INST_DPD_MEAN           = ('DPD',           'mean'),  # avg days late
        INST_DPD_MAX            = ('DPD',           'max'),   # worst lateness ever
        INST_DPD_SUM            = ('DPD',           'sum'),   # total days late
        INST_DBD_MEAN           = ('DBD',           'mean'),  # avg days early
        INST_NUM_LATE           = ('DPD',           lambda x: (x > 0).sum()),  # count of late payments
    ).reset_index()

    # Ratio of late payments to total payments
    inst_agg['INST_LATE_PAYMENT_RATE'] = (
        inst_agg['INST_NUM_LATE'] / (inst_agg['INST_COUNT'] + 1)
    )

    df = df.merge(inst_agg, on='SK_ID_CURR', how='left')
    print(f"  After installments merge: {df.shape}")

except FileNotFoundError:
    print("  installments_payments.csv not found — skipping")



  installments_payments: (13605401, 8)
  After installments merge: (307511, 170)


In [10]:
# credit_card_balance.csv  (~+0.003 AUC)
try:
    cc = pd.read_csv('credit_card_balance.csv')
    cc_agg = cc.groupby('SK_ID_CURR').agg(
        CC_AMT_BALANCE_MEAN      = ('AMT_BALANCE',          'mean'),
        CC_AMT_CREDIT_LIMIT_MEAN = ('AMT_CREDIT_LIMIT_ACTUAL','mean'),
        CC_CNT_DRAWINGS_MEAN     = ('CNT_DRAWINGS_CURRENT', 'mean'),
        CC_SK_DPD_MAX            = ('SK_DPD',                'max'),
    ).reset_index()
    cc_agg['CC_UTILIZATION'] = (
        cc_agg['CC_AMT_BALANCE_MEAN'] /
        (cc_agg['CC_AMT_CREDIT_LIMIT_MEAN'] + 1)
    )
    df = df.merge(cc_agg, on='SK_ID_CURR', how='left')
except FileNotFoundError:
    print("credit_card_balance.csv not found — skipping")

In [11]:
missing_ratio = df.isnull().mean()
cols_to_drop  = missing_ratio[missing_ratio > 0.6].index
df = df.drop(columns=cols_to_drop)

if 'DAYS_EMPLOYED' in df.columns:
    df['DAYS_EMPLOYED_ANOM'] = (df['DAYS_EMPLOYED'] == 365243).astype(int)
    df['DAYS_EMPLOYED']      = df['DAYS_EMPLOYED'].replace(365243, np.nan)

num_cols = df.select_dtypes(include=['number']).columns
cat_cols = df.select_dtypes(include=['object']).columns

df[num_cols] = df[num_cols].fillna(df[num_cols].median())
for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])


In [12]:
df['CREDIT_INCOME_RATIO']  = df['AMT_CREDIT']  / (df['AMT_INCOME_TOTAL'] + 1)
df['ANNUITY_INCOME_RATIO'] = df['AMT_ANNUITY'] / (df['AMT_INCOME_TOTAL'] + 1)
df['CREDIT_TERM']          = df['AMT_ANNUITY'] / (df['AMT_CREDIT'] + 1)
df['INCOME_PER_PERSON']    = df['AMT_INCOME_TOTAL'] / (df['CNT_FAM_MEMBERS'] + 1)

if 'AMT_GOODS_PRICE' in df.columns:
    df['GOODS_PRICE_CREDIT_DIFF'] = df['AMT_CREDIT'] - df['AMT_GOODS_PRICE']

if 'DAYS_BIRTH' in df.columns:
    df['AGE_YEARS'] = -df['DAYS_BIRTH'] / 365

if 'DAYS_EMPLOYED' in df.columns and 'DAYS_BIRTH' in df.columns:
    df['EMPLOYMENT_TO_AGE_RATIO'] = df['DAYS_EMPLOYED'] / (df['DAYS_BIRTH'] + 1)

print(f"Final feature count : {df.shape[1]}")

Final feature count : 156


In [13]:
le = LabelEncoder()
for col in cat_cols:
    if col in df.columns:
        df[col] = le.fit_transform(df[col].astype(str))

# ============================================================
# STEP 7 — TRAIN / VAL SPLIT
# ============================================================
TARGET = 'TARGET'
X = df.drop(columns=[TARGET, 'SK_ID_CURR'], errors='ignore')
y = df[TARGET]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scale_pos_weight = float((y_train == 0).sum()) / float((y_train == 1).sum())
print(f"scale_pos_weight = {scale_pos_weight:.2f} | "
      f"Train: {X_train.shape} | Val: {X_val.shape}")


scale_pos_weight = 11.39 | Train: (246008, 154) | Val: (61503, 154)


In [14]:
N_TRIALS = 100
CV_FOLDS  = 5

def objective(trial):
    n_estimators      = trial.suggest_int  ('n_estimators',      300, 1500, step=100)
    max_depth         = trial.suggest_int  ('max_depth',         3, 9)
    min_child_weight  = trial.suggest_int  ('min_child_weight',  1, 10)
    learning_rate     = trial.suggest_float('learning_rate',     0.005, 0.2,  log=True)
    subsample         = trial.suggest_float('subsample',         0.5,  1.0)
    colsample_bytree  = trial.suggest_float('colsample_bytree',  0.4,  1.0)
    colsample_bylevel = trial.suggest_float('colsample_bylevel', 0.4,  1.0)  # ✅ new
    reg_alpha         = trial.suggest_float('reg_alpha',         1e-8, 10.0, log=True)
    reg_lambda        = trial.suggest_float('reg_lambda',        1e-8, 10.0, log=True)
    gamma             = trial.suggest_float('gamma',             0.0,  5.0)
    min_split_loss    = trial.suggest_float('min_split_loss',    0.0,  2.0)  # ✅ new

    cv         = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=42)
    auc_scores = []

    for train_idx, valid_idx in cv.split(X_train, y_train):
        Xtr = X_train.iloc[train_idx]
        Xvl = X_train.iloc[valid_idx]
        ytr = y_train.iloc[train_idx]
        yvl = y_train.iloc[valid_idx]

        from xgboost import XGBClassifier
        model = XGBClassifier(
            n_estimators          = n_estimators,
            max_depth             = max_depth,
            min_child_weight      = min_child_weight,
            learning_rate         = learning_rate,
            subsample             = subsample,
            colsample_bytree      = colsample_bytree,
            colsample_bylevel     = colsample_bylevel,  # ✅ new
            reg_alpha             = reg_alpha,
            reg_lambda            = reg_lambda,
            gamma                 = gamma,
            min_split_loss        = min_split_loss,     # ✅ new
            scale_pos_weight      = scale_pos_weight,
            eval_metric           = 'auc',
            early_stopping_rounds = 30,
            random_state          = 42,
            n_jobs                = -1,
            tree_method           = 'hist',
            device      = 'cuda',
        )
        model.fit(Xtr, ytr, eval_set=[(Xvl, yvl)], verbose=False)
        auc_scores.append(roc_auc_score(yvl, model.predict_proba(Xvl)[:, 1]))

    return float(np.mean(auc_scores))


print(f"\nRunning Optuna: {N_TRIALS} trials × {CV_FOLDS}-fold CV ...")
study = optuna.create_study(
    direction = 'maximize',
    sampler   = optuna.samplers.TPESampler(seed=42),
)
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

print(f"\n✅ Best CV ROC-AUC : {study.best_value:.6f}")
print(f"   Best params     : {study.best_params}")
bp = study.best_params


Running Optuna: 100 trials × 5-fold CV ...


  0%|          | 0/100 [00:00<?, ?it/s]


✅ Best CV ROC-AUC : 0.780743
   Best params     : {'n_estimators': 1500, 'max_depth': 4, 'min_child_weight': 7, 'learning_rate': 0.02771266433624557, 'subsample': 0.8493250000369008, 'colsample_bytree': 0.9941300711384321, 'colsample_bylevel': 0.6436127039338319, 'reg_alpha': 9.726704417829254, 'reg_lambda': 0.2375733123483753, 'gamma': 4.654051602954547, 'min_split_loss': 0.22091121044652268}


In [15]:
from xgboost import XGBClassifier

final_model = XGBClassifier(
    n_estimators          = bp['n_estimators'],
    max_depth             = bp['max_depth'],
    min_child_weight      = bp['min_child_weight'],
    learning_rate         = bp['learning_rate'],
    subsample             = bp['subsample'],
    colsample_bytree      = bp['colsample_bytree'],
    colsample_bylevel     = bp['colsample_bylevel'],   # ✅ added
    reg_alpha             = bp['reg_alpha'],
    reg_lambda            = bp['reg_lambda'],
    gamma                 = bp['gamma'],
    min_split_loss        = bp['min_split_loss'],       # ✅ added
    scale_pos_weight      = scale_pos_weight,
    eval_metric           = 'auc',
    early_stopping_rounds = 50,
    random_state          = 42,
    n_jobs                = -1,
    tree_method           = 'hist',
    device      = 'cuda',
)

final_model.fit(
    X_train, y_train,
    eval_set = [(X_val, y_val)],
    verbose  = 100,
)

print(f"\n✅ Final model trained")
print(f"   Best iteration : {final_model.best_iteration}")
print(f"   Best AUC score : {final_model.best_score:.6f}")

[0]	validation_0-auc:0.70034
[100]	validation_0-auc:0.75653
[200]	validation_0-auc:0.76952
[300]	validation_0-auc:0.77454
[400]	validation_0-auc:0.77701
[500]	validation_0-auc:0.77880
[600]	validation_0-auc:0.78035
[700]	validation_0-auc:0.78159
[800]	validation_0-auc:0.78263
[900]	validation_0-auc:0.78335
[1000]	validation_0-auc:0.78414
[1100]	validation_0-auc:0.78482
[1200]	validation_0-auc:0.78523
[1300]	validation_0-auc:0.78549
[1400]	validation_0-auc:0.78568
[1499]	validation_0-auc:0.78602

✅ Final model trained
   Best iteration : 1499
   Best AUC score : 0.786019


In [16]:
y_proba = final_model.predict_proba(X_val)[:, 1]
roc_auc = roc_auc_score(y_val, y_proba)
print(f"\n🎯 Validation ROC-AUC: {roc_auc:.6f}")

precisions, recalls, thresholds = precision_recall_curve(y_val, y_proba)
f1_scores   = 2 * precisions * recalls / (precisions + recalls + 1e-9)
best_thresh = float(thresholds[np.argmax(f1_scores)])
print(f"   Optimal threshold : {best_thresh:.4f}")

print("\n── Default threshold (0.50) ──")
print(classification_report(y_val, (y_proba >= 0.50).astype(int)))

print(f"\n── Optimal threshold ({best_thresh:.4f}) ──")
print(classification_report(y_val, (y_proba >= best_thresh).astype(int)))


🎯 Validation ROC-AUC: 0.786019
   Optimal threshold : 0.6666

── Default threshold (0.50) ──
              precision    recall  f1-score   support

           0       0.96      0.74      0.84     56538
           1       0.19      0.68      0.30      4965

    accuracy                           0.74     61503
   macro avg       0.58      0.71      0.57     61503
weighted avg       0.90      0.74      0.80     61503


── Optimal threshold (0.6666) ──
              precision    recall  f1-score   support

           0       0.95      0.90      0.92     56538
           1       0.28      0.44      0.34      4965

    accuracy                           0.86     61503
   macro avg       0.61      0.67      0.63     61503
weighted avg       0.89      0.86      0.88     61503



In [17]:
feat_imp = pd.Series(final_model.feature_importances_, index=X_train.columns)
print("\nTop 20 features:")
print(feat_imp.nlargest(20).to_string())



Top 20 features:
EXT_SOURCE_2                0.062794
EXT_SOURCE_3                0.054777
NAME_EDUCATION_TYPE         0.034075
CODE_GENDER                 0.024012
EXT_SOURCE_1                0.020732
DAYS_EMPLOYED               0.020287
INST_LATE_PAYMENT_RATE      0.018299
GOODS_PRICE_CREDIT_DIFF     0.017359
FLAG_OWN_CAR                0.015648
FLAG_DOCUMENT_3             0.015455
INST_PAYMENT_RATIO_MEAN     0.014974
PREV_APPROVAL_RATE          0.013981
PREV_REFUSED_COUNT          0.013969
NAME_INCOME_TYPE            0.011760
BUREAU_DEBT_CREDIT_RATIO    0.011516
REG_CITY_NOT_LIVE_CITY      0.011449
CREDIT_TERM                 0.011077
NAME_CONTRACT_TYPE          0.010945
AGE_YEARS                   0.010834
INST_DPD_MEAN               0.010633


In [18]:
joblib.dump(final_model, 'xgb_credit_risk_v3.pkl')
print("\n💾 Model saved → xgb_credit_risk_v3.pkl")



💾 Model saved → xgb_credit_risk_v3.pkl


In [21]:
# ── Load test set ──
df_test = pd.read_csv('application_test.csv')
test_ids = df_test['SK_ID_CURR'].copy()
print(f"Test set: {df_test.shape}")

# ── Apply all same merges as training ──

# Bureau + bureau_balance
try:
    bureau = pd.read_csv('bureau.csv')
    try:
        bb = pd.read_csv('bureau_balance.csv')
        bb_agg = bb.groupby('SK_ID_BUREAU').agg(
            BB_MONTHS_COUNT     = ('MONTHS_BALANCE', 'count'),
            BB_STATUS_C_COUNT   = ('STATUS', lambda x: (x == 'C').sum()),
            BB_STATUS_X_COUNT   = ('STATUS', lambda x: (x == 'X').sum()),
            BB_STATUS_BAD_COUNT = ('STATUS', lambda x: x.isin(['1','2','3','4','5']).sum()),
        ).reset_index()
        bb_agg['BB_BAD_RATE'] = bb_agg['BB_STATUS_BAD_COUNT'] / (bb_agg['BB_MONTHS_COUNT'] + 1)
        bureau = bureau.merge(bb_agg, on='SK_ID_BUREAU', how='left')
    except FileNotFoundError:
        pass
    bureau_agg = bureau.groupby('SK_ID_CURR').agg(
        BUREAU_LOAN_COUNT          = ('SK_ID_BUREAU',        'count'),
        BUREAU_ACTIVE_COUNT        = ('CREDIT_ACTIVE',       lambda x: (x == 'Active').sum()),
        BUREAU_CLOSED_COUNT        = ('CREDIT_ACTIVE',       lambda x: (x == 'Closed').sum()),
        BUREAU_AMT_CREDIT_SUM      = ('AMT_CREDIT_SUM',      'sum'),
        BUREAU_AMT_CREDIT_DEBT_SUM = ('AMT_CREDIT_SUM_DEBT', 'sum'),
        BUREAU_AMT_CREDIT_OVERDUE  = ('AMT_CREDIT_SUM_OVERDUE', 'sum'),
        BUREAU_DAYS_CREDIT_MEAN    = ('DAYS_CREDIT',         'mean'),
        BUREAU_DAYS_CREDIT_MAX     = ('DAYS_CREDIT',         'max'),
        BUREAU_DAYS_ENDDATE_MEAN   = ('DAYS_CREDIT_ENDDATE', 'mean'),
        BUREAU_CNT_PROLONG_SUM     = ('CNT_CREDIT_PROLONG',  'sum'),
        BB_MONTHS_COUNT_MEAN       = ('BB_MONTHS_COUNT',     'mean'),
        BB_BAD_COUNT_MEAN          = ('BB_STATUS_BAD_COUNT', 'mean'),
        BB_BAD_COUNT_SUM           = ('BB_STATUS_BAD_COUNT', 'sum'),
        BB_BAD_RATE_MEAN           = ('BB_BAD_RATE',         'mean'),
        BB_BAD_RATE_MAX            = ('BB_BAD_RATE',         'max'),
    ).reset_index()
    bureau_agg['BUREAU_DEBT_CREDIT_RATIO'] = (
        bureau_agg['BUREAU_AMT_CREDIT_DEBT_SUM'] / (bureau_agg['BUREAU_AMT_CREDIT_SUM'] + 1)
    )
    bureau_agg['BUREAU_ACTIVE_RATIO'] = (
        bureau_agg['BUREAU_ACTIVE_COUNT'] / (bureau_agg['BUREAU_LOAN_COUNT'] + 1)
    )
    df_test = df_test.merge(bureau_agg, on='SK_ID_CURR', how='left')
except FileNotFoundError:
    pass

# POS_CASH_balance
try:
    pos = pd.read_csv('POS_CASH_balance.csv')
    pos_agg = pos.groupby('SK_ID_CURR').agg(
        POS_MONTHS_BALANCE_MEAN = ('MONTHS_BALANCE',       'mean'),
        POS_CNT_INSTALMENT_MEAN = ('CNT_INSTALMENT',       'mean'),
        POS_SK_DPD_MAX          = ('SK_DPD',               'max'),
        POS_SK_DPD_DEF_SUM      = ('SK_DPD_DEF',           'sum'),
        POS_COMPLETED_COUNT     = ('NAME_CONTRACT_STATUS', lambda x: (x == 'Completed').sum()),
    ).reset_index()
    df_test = df_test.merge(pos_agg, on='SK_ID_CURR', how='left')
except FileNotFoundError:
    pass

# Previous application
try:
    prev = pd.read_csv('previous_application.csv')
    prev_agg = prev.groupby('SK_ID_CURR').agg(
        PREV_APP_COUNT             = ('SK_ID_PREV',           'count'),
        PREV_APPROVED_COUNT        = ('NAME_CONTRACT_STATUS', lambda x: (x == 'Approved').sum()),
        PREV_REFUSED_COUNT         = ('NAME_CONTRACT_STATUS', lambda x: (x == 'Refused').sum()),
        PREV_AMT_CREDIT_MEAN       = ('AMT_CREDIT',           'mean'),
        PREV_AMT_ANNUITY_MEAN      = ('AMT_ANNUITY',          'mean'),
        PREV_AMT_DOWN_PAYMENT_MEAN = ('AMT_DOWN_PAYMENT',     'mean'),
        PREV_DAYS_DECISION_MEAN    = ('DAYS_DECISION',        'mean'),
        PREV_DAYS_DECISION_MIN     = ('DAYS_DECISION',        'min'),
        PREV_RATE_DOWN_PAYMENT_MEAN= ('RATE_DOWN_PAYMENT',    'mean'),
    ).reset_index()
    prev_agg['PREV_APPROVAL_RATE'] = (
        prev_agg['PREV_APPROVED_COUNT'] / (prev_agg['PREV_APP_COUNT'] + 1)
    )
    df_test = df_test.merge(prev_agg, on='SK_ID_CURR', how='left')
except FileNotFoundError:
    pass

# Installments
try:
    inst = pd.read_csv('installments_payments.csv')
    inst['DPD'] = (inst['DAYS_ENTRY_PAYMENT'] - inst['DAYS_INSTALMENT']).clip(lower=0)
    inst['DBD'] = (inst['DAYS_INSTALMENT'] - inst['DAYS_ENTRY_PAYMENT']).clip(lower=0)
    inst['PAYMENT_DIFF']  = inst['AMT_INSTALMENT'] - inst['AMT_PAYMENT']
    inst['PAYMENT_RATIO'] = inst['AMT_PAYMENT'] / (inst['AMT_INSTALMENT'] + 1)
    inst_agg = inst.groupby('SK_ID_CURR').agg(
        INST_COUNT              = ('SK_ID_PREV',    'count'),
        INST_AMT_PAYMENT_SUM    = ('AMT_PAYMENT',   'sum'),
        INST_AMT_PAYMENT_MEAN   = ('AMT_PAYMENT',   'mean'),
        INST_PAYMENT_DIFF_MEAN  = ('PAYMENT_DIFF',  'mean'),
        INST_PAYMENT_DIFF_MAX   = ('PAYMENT_DIFF',  'max'),
        INST_PAYMENT_RATIO_MEAN = ('PAYMENT_RATIO', 'mean'),
        INST_DPD_MEAN           = ('DPD',           'mean'),
        INST_DPD_MAX            = ('DPD',           'max'),
        INST_DPD_SUM            = ('DPD',           'sum'),
        INST_DBD_MEAN           = ('DBD',           'mean'),
        INST_NUM_LATE           = ('DPD',           lambda x: (x > 0).sum()),
    ).reset_index()
    inst_agg['INST_LATE_PAYMENT_RATE'] = (
        inst_agg['INST_NUM_LATE'] / (inst_agg['INST_COUNT'] + 1)
    )
    df_test = df_test.merge(inst_agg, on='SK_ID_CURR', how='left')
except FileNotFoundError:
    pass

# Credit card
try:
    cc = pd.read_csv('credit_card_balance.csv')
    cc_agg = cc.groupby('SK_ID_CURR').agg(
        CC_AMT_BALANCE_MEAN      = ('AMT_BALANCE',             'mean'),
        CC_AMT_CREDIT_LIMIT_MEAN = ('AMT_CREDIT_LIMIT_ACTUAL', 'mean'),
        CC_CNT_DRAWINGS_MEAN     = ('CNT_DRAWINGS_CURRENT',    'mean'),
        CC_SK_DPD_MAX            = ('SK_DPD',                  'max'),
    ).reset_index()
    cc_agg['CC_UTILIZATION'] = (
        cc_agg['CC_AMT_BALANCE_MEAN'] / (cc_agg['CC_AMT_CREDIT_LIMIT_MEAN'] + 1)
    )
    df_test = df_test.merge(cc_agg, on='SK_ID_CURR', how='left')
except FileNotFoundError:
    pass

# ── Apply same cleaning ──
if 'DAYS_EMPLOYED' in df_test.columns:
    df_test['DAYS_EMPLOYED_ANOM'] = (df_test['DAYS_EMPLOYED'] == 365243).astype(int)
    df_test['DAYS_EMPLOYED']      = df_test['DAYS_EMPLOYED'].replace(365243, np.nan)

num_cols_t = df_test.select_dtypes(include=['number']).columns
cat_cols_t  = df_test.select_dtypes(include=['object']).columns
df_test[num_cols_t] = df_test[num_cols_t].fillna(df_test[num_cols_t].median())
for col in cat_cols_t:
    df_test[col] = df_test[col].fillna(df_test[col].mode()[0])

# ── Apply same feature engineering ──
df_test['CREDIT_INCOME_RATIO']  = df_test['AMT_CREDIT']  / (df_test['AMT_INCOME_TOTAL'] + 1)
df_test['ANNUITY_INCOME_RATIO'] = df_test['AMT_ANNUITY'] / (df_test['AMT_INCOME_TOTAL'] + 1)
df_test['CREDIT_TERM']          = df_test['AMT_ANNUITY'] / (df_test['AMT_CREDIT'] + 1)
df_test['INCOME_PER_PERSON']    = df_test['AMT_INCOME_TOTAL'] / (df_test['CNT_FAM_MEMBERS'] + 1)
if 'AMT_GOODS_PRICE' in df_test.columns:
    df_test['GOODS_PRICE_CREDIT_DIFF'] = df_test['AMT_CREDIT'] - df_test['AMT_GOODS_PRICE']
if 'DAYS_BIRTH' in df_test.columns:
    df_test['AGE_YEARS'] = -df_test['DAYS_BIRTH'] / 365
if 'DAYS_EMPLOYED' in df_test.columns and 'DAYS_BIRTH' in df_test.columns:
    df_test['EMPLOYMENT_TO_AGE_RATIO'] = df_test['DAYS_EMPLOYED'] / (df_test['DAYS_BIRTH'] + 1)

# ── Label encode categoricals ──
for col in cat_cols_t:
    if col in df_test.columns:
        df_test[col] = le.fit_transform(df_test[col].astype(str))

# ── CRITICAL: align columns exactly to training ──
X_test = df_test.drop(columns=['SK_ID_CURR', 'TARGET'], errors='ignore')
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

# ── Predict and save ──
test_proba = final_model.predict_proba(X_test)[:, 1]
submission = pd.DataFrame({'SK_ID_CURR': test_ids, 'TARGET': test_proba})
submission.to_csv('submission.csv', index=False)

print(f"✅ submission.csv saved — {len(submission)} rows")
print(f"   Predicted default rate : {(test_proba > 0.5).mean():.2%}")
print(f"   Score range            : {test_proba.min():.4f} – {test_proba.max():.4f}")

Test set: (48744, 121)
✅ submission.csv saved — 48744 rows
   Predicted default rate : 24.29%
   Score range            : 0.0051 – 0.9692


In [22]:
sub = pd.read_csv('submission.csv')
print(sub['TARGET'].describe())
print(f"\nUnique values sample: {sub['TARGET'].unique()[:10]}")

count    48744.000000
mean         0.343004
std          0.214291
min          0.005051
25%          0.165747
50%          0.300203
75%          0.493013
max          0.969230
Name: TARGET, dtype: float64

Unique values sample: [0.24326262 0.6707361  0.11881496 0.38005438 0.4576065  0.19433643
 0.12867908 0.1882592  0.07046802 0.47245097]
